# Feature Engineering

This notebook performs following tasks-
- Raw data is loaded from an Athena table.
- Feature engineering techniques are applied—including binary label encoding, feature creation , and scaling.
-  Random Forest classifier is used to select top 10 features.
-  Final engineered dataset is saved as a CSV and uploaded to an S3 bucket.
-  A new Athena table is registered to enable querying this refined dataset.


## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import boto3
from pyathena import connect
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import sys
import os
import io

In [2]:
# Read the config
sys.path.append('../config')
import config

bucket = config.S3_BUCKET
s3_staging_prefix = config.ATHENA_STAGING_PREFIX
s3_staging_path = f's3://{bucket}/{s3_staging_prefix}/'
db = config.ATHENA_DB_NAME
table = config.DATA_TABLE_NAME

# Set the feature store path
s3_engineered_path = f's3://{bucket}/final_project/engineered/'
engineered_filename = 'engineered_features.csv'

#print("Using bucket:", bucket, "Table:", s3_staging_path)

In [4]:
# Setup AWS session
session = boto3.session.Session()
region = session.region_name

# start s3 client
s3_client = session.client('s3', region_name=region)

In [5]:
# List files in the prefix
prefix = 'final_project/staging/'
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
'''
# Display
if 'Contents' in response:
    print(f"📂 Objects under s3://{bucket}/{s3_staging_path}:")
    for obj in response['Contents']:
        print(" -", obj['Key'])
else:
    print("No objects found.")'''

'\n# Display\nif \'Contents\' in response:\n    print(f"📂 Objects under s3://{bucket}/{s3_staging_path}:")\n    for obj in response[\'Contents\']:\n        print(" -", obj[\'Key\'])\nelse:\n    print("No objects found.")'

# Feature engineering

In [6]:
# Connect to staging directory and read table

conn = connect(region_name=region, s3_staging_dir=s3_staging_path)
query = f"SELECT * FROM {db}.{table}"
df = pd.read_sql(query, conn)

/tmp/ipykernel_141/1071780348.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [7]:
df.head()

,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [8]:
print(df.shape, df.columns, df.dtypes)

(221264, 78) Index(['protocol', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'fwd_packets_length_total',
       'bwd_packets_length_total', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_bytes_per_s', 'flow_packets_per_s',
       'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max',
       'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std',
       'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags',
       'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets_per_s', 'bwd_packets_per_s',
       'packet_length_min', 'packet_length_max', 'packet_length_mean',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
    

In [9]:
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 1])

flow_bytes_per_s        221264
flow_packets_per_s      221264
fwd_packets_per_s       221264
bwd_packets_per_s       221264
down_up_ratio           221264
fwd_avg_bytes_bulk      221264
fwd_avg_packets_bulk    221264
bwd_avg_bytes_bulk      221264
bwd_avg_packets_bulk    221264
dtype: int64


In [10]:
df.shape

(221264, 78)

In [11]:
# Drop columns with all NaNs
df_cleaned = df.dropna(axis=1, thresh=1)

In [12]:
df_cleaned.shape

(221264, 69)

In [13]:
# Binary label encoding
df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)

/tmp/ipykernel_141/2488267272.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)


In [14]:
print(df_cleaned.head())
print(df_cleaned.shape, df_cleaned.columns, df_cleaned.dtypes)

   protocol  flow_duration  total_fwd_packets  total_backward_packets  \
0         6              3                  2                       0   
1         6            109                  1                       1   
2         6             52                  1                       1   
3         6             34                  1                       1   
4         6              3                  2                       0   

   fwd_packets_length_total  bwd_packets_length_total  fwd_packet_length_max  \
0                        12                         0                      6   
1                         6                         6                      6   
2                         6                         6                      6   
3                         6                         6                      6   
4                        12                         0                      6   

   fwd_packet_length_min  fwd_packet_length_mean  fwd_packet_length_std  ...  \


In [15]:
# Normalize numeric features
features = df.drop('label', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_scaled_df = pd.DataFrame(X_scaled, columns=features.columns)
X_scaled_df['label'] = df['label']

/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [16]:
# Feature selection using Random Forest
X = X_scaled_df.drop('label', axis=1)
y = X_scaled_df['label']
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

RandomForestClassifier(random_state=42)

In [17]:
# Select top 10 features
importances = clf.feature_importances_
top_features = X.columns[np.argsort(importances)[::-1][:10]]


In [18]:
# df_final dataframe with final selected features will be stored. df_final can be used by the model to train/validate and test.
df_final = df_cleaned[top_features.tolist() + ['label']]

In [19]:
print(top_features)

Index(['fwd_packet_length_mean', 'subflow_fwd_bytes', 'fwd_packet_length_max',
       'fwd_packets_length_total', 'fwd_act_data_packets',
       'avg_fwd_segment_size', 'fwd_iat_std', 'init_fwd_win_bytes',
       'subflow_fwd_packets', 'fwd_header_length'],
      dtype='object')


In [20]:
# Append feature output path to config.py
with open('../config/config.py', 'a') as f:
    f.write(f"FEATURE_ENGINEERED_PATH = 's3://{bucket}/final_project/engineered/{engineered_filename}'\n")    

## Ingest Data into FeatureStore

### Feature store setup

In [42]:
from sagemaker.session import Session
sagemaker_client = session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

### Define FeatureGroups

In [43]:
import time
from time import gmtime, strftime, sleep

intrusion_feature_group_name = "intrusion-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

In [44]:
intrusion_feature_group_name

'intrusion-feature-group-27-04-21-35'

In [45]:
from sagemaker.feature_store.feature_group import FeatureGroup

intrusion_feature_group = FeatureGroup(
    name=intrusion_feature_group_name, sagemaker_session=feature_store_session
)


In [48]:
import uuid
current_time_sec = int(round(time.time()))

# Move target column to front
target_col = "label"
df_final = df_final[[target_col] + [col for col in df_final.columns if col != target_col]]

# Add required metadata columns
record_identifier_name = "record_id"
df_final[record_identifier_name] = [str(uuid.uuid4()) for _ in range(len(df_final))]
event_time_feature_name = 'event_time'
df_final['event_time'] = pd.Series([current_time_sec] * len(df_final), dtype="float64")

In [49]:
# load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
intrusion_feature_group.load_feature_definitions(data_frame=df_final)

[FeatureDefinition(feature_name='label', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_packet_length_mean', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='subflow_fwd_bytes', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_packet_length_max', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_packets_length_total', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_act_data_packets', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='avg_fwd_segment_size', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_iat_std', feature_type=<FeatureTypeEnum.FR

### Create FeatureGroups in SageMaker FeatureStore

In [50]:
from sagemaker import get_execution_role

# You can modify the following to use a role of your choosing. See the documentation for how to create this.
role = get_execution_role()
print(role)

arn:aws:iam::249645693565:role/LabRole


In [51]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


intrusion_feature_group.create(
    s3_uri= s3_engineered_path,
    record_identifier_name= record_identifier_feature_name,
    event_time_feature_name= event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)


wait_for_feature_group_creation_complete(feature_group=intrusion_feature_group)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup intrusion-feature-group-27-04-21-35 successfully created.


In [52]:
# Confirm the FeatureGroup has been created by using the DescribeFeatureGroup and ListFeatureGroups APIs.
intrusion_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:249645693565:feature-group/intrusion-feature-group-27-04-21-35',
 'FeatureGroupName': 'intrusion-feature-group-27-04-21-35',
 'RecordIdentifierFeatureName': 'label',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'label', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_packet_length_mean', 'FeatureType': 'Fractional'},
  {'FeatureName': 'subflow_fwd_bytes', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_packet_length_max', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_packets_length_total', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_act_data_packets', 'FeatureType': 'Integral'},
  {'FeatureName': 'avg_fwd_segment_size', 'FeatureType': 'Fractional'},
  {'FeatureName': 'fwd_iat_std', 'FeatureType': 'Fractional'},
  {'FeatureName': 'init_fwd_win_bytes', 'FeatureType': 'Integral'},
  {'FeatureName': 'subflow_fwd_packets', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_header_length', 'F

In [53]:
sagemaker_client.list_feature_groups()  # use boto client to list FeatureGroups

{'FeatureGroupSummaries': [{'FeatureGroupName': 'neighborhood-feature-group-21-22-21-05',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:249645693565:feature-group/neighborhood-feature-group-21-22-21-05',
   'CreationTime': datetime.datetime(2025, 5, 21, 22, 21, 5, 620000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}},
  {'FeatureGroupName': 'neighborhood-feature-group-21-21-46-40',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:249645693565:feature-group/neighborhood-feature-group-21-21-46-40',
   'CreationTime': datetime.datetime(2025, 5, 21, 21, 46, 51, 564000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}},
  {'FeatureGroupName': 'intrusion-feature-group-27-04-21-35',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:249645693565:feature-group/intrusion-feature-group-27-04-21-35',
   'CreationTime': datetime.datetime(2025, 5, 27, 4, 22, 54, 17000, tzinfo=tzlocal()

In [54]:
prefix_fs = 'final_project/engineered/'

In [55]:
# Convert DataFrame to CSV in memory
csv_buffer = io.StringIO()
df_final.to_csv(csv_buffer, index=False)

# Upload to S3
s3_client.put_object(
    Bucket=bucket,
    Key=f'{prefix_fs}/{engineered_filename}',
    Body=csv_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': '7JP3KRE3Q0PVV8XD',
  'HostId': 'gO/IjP8a+Tgfj68tG19YVtiKRnlDT0A0hqnsPPuv97TJU4oLNMbN4PkjdieokrDKFTTOMzI4UpjKXnFNXbQPXg619xZ72s4I',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'gO/IjP8a+Tgfj68tG19YVtiKRnlDT0A0hqnsPPuv97TJU4oLNMbN4PkjdieokrDKFTTOMzI4UpjKXnFNXbQPXg619xZ72s4I',
   'x-amz-request-id': '7JP3KRE3Q0PVV8XD',
   'date': 'Tue, 27 May 2025 04:23:26 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"2f1c79f8145e90fdcaa8e1204e0d0a86"',
   'x-amz-checksum-crc32': 'dGGboQ==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"2f1c79f8145e90fdcaa8e1204e0d0a86"',
 'ChecksumCRC32': 'dGGboQ==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

### PutRecords into FeatureGroup

In [56]:
intrusion_feature_group.ingest(data_frame=df_final, max_workers=3, wait=True)

IngestionManagerPandas(feature_group_name='intrusion-feature-group-27-04-21-35', feature_definitions={'label': {'FeatureName': 'label', 'FeatureType': 'Integral'}, 'fwd_packet_length_mean': {'FeatureName': 'fwd_packet_length_mean', 'FeatureType': 'Fractional'}, 'subflow_fwd_bytes': {'FeatureName': 'subflow_fwd_bytes', 'FeatureType': 'Integral'}, 'fwd_packet_length_max': {'FeatureName': 'fwd_packet_length_max', 'FeatureType': 'Integral'}, 'fwd_packets_length_total': {'FeatureName': 'fwd_packets_length_total', 'FeatureType': 'Integral'}, 'fwd_act_data_packets': {'FeatureName': 'fwd_act_data_packets', 'FeatureType': 'Integral'}, 'avg_fwd_segment_size': {'FeatureName': 'avg_fwd_segment_size', 'FeatureType': 'Fractional'}, 'fwd_iat_std': {'FeatureName': 'fwd_iat_std', 'FeatureType': 'Fractional'}, 'init_fwd_win_bytes': {'FeatureName': 'init_fwd_win_bytes', 'FeatureType': 'Integral'}, 'subflow_fwd_packets': {'FeatureName': 'subflow_fwd_packets', 'FeatureType': 'Integral'}, 'fwd_header_length

## TESTING

### Test 1 - Get All Records as a DataFrame using Athena query

In [57]:
intrusion_feature_group_name

'intrusion-feature-group-27-04-21-35'

In [58]:
response = sagemaker_client.describe_feature_group(FeatureGroupName=intrusion_feature_group_name)
offline_store_uri = response["OfflineStoreConfig"]["S3StorageConfig"]["S3Uri"]
print("📂 Offline store S3 location:", offline_store_uri)


📂 Offline store S3 location: s3://sagemaker-us-east-1-249645693565/final_project/engineered/


In [ ]:
# Query via Athena Using PyAthena
from pyathena import connect

### Test2- load dataframe from csv file


In [69]:

response = s3_client.list_objects_v2(Bucket=bucket, Prefix='final_project/engineered')

In [71]:
# Filter for .csv files
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]
'''
if csv_files:
    print("Found CSV files:")
    for file in csv_files:
        print("-", file)
else:
    print("No CSV files found.")'''

'\nif csv_files:\n    print("Found CSV files:")\n    for file in csv_files:\n        print("-", file)\nelse:\n    print("No CSV files found.")'

In [73]:
# Test 

# Choose first CSV file from list
key = csv_files[0]

# Load it into memory
obj = s3_client.get_object(Bucket=bucket, Key=key)
df_features = pd.read_csv(io.BytesIO(obj['Body'].read()))

# Preview it
print(f"📄 Preview of {key}")



📄 Preview of final_project/engineered//engineered_features.csv


In [74]:
print(df_features.head())
print(df_features.shape)
print(df_features.columns)
print(df_features.dtypes)

   label  fwd_packet_length_mean  subflow_fwd_bytes  fwd_packet_length_max  \
0      0                     6.0                 12                      6   
1      0                     6.0                  6                      6   
2      0                     6.0                  6                      6   
3      0                     6.0                  6                      6   
4      0                     6.0                 12                      6   

   fwd_packets_length_total  fwd_act_data_packets  avg_fwd_segment_size  \
0                        12                     1                   6.0   
1                         6                     0                   6.0   
2                         6                     0                   6.0   
3                         6                     0                   6.0   
4                        12                     1                   6.0   

   fwd_iat_std  init_fwd_win_bytes  subflow_fwd_packets  fwd_header_length  \
0 

In [75]:
df_features.columns

Index(['label', 'fwd_packet_length_mean', 'subflow_fwd_bytes',
       'fwd_packet_length_max', 'fwd_packets_length_total',
       'fwd_act_data_packets', 'avg_fwd_segment_size', 'fwd_iat_std',
       'init_fwd_win_bytes', 'subflow_fwd_packets', 'fwd_header_length',
       'event_time', 'record_id'],
      dtype='object')

In [77]:
print("Feature Engineered file:", engineered_filename, "Stored.")

Feature Engineered file: engineered_features.csv Stored.


In [78]:
print(f"Feature-engineering completed.")

Feature-engineering completed.
